# Attention Residual Contribution

How attention outputs shape the residual stream:
alignment, per-head contributions, update magnitude,
and direction consistency.

## Why This Matters

Attention residual contribution reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_residual_contribution import (
    attention_residual_alignment, per_head_residual_contribution,
    attention_update_magnitude, attention_direction_consistency,
    attention_residual_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Attention-Residual Alignment

Does attention reinforce the existing representation or add orthogonal info?

In [ ]:
for layer in range(4):
    result = attention_residual_alignment(model, tokens, layer=layer)
    flag = ' [REINFORCING]' if result['is_reinforcing'] else ''
    print(f"  Layer {layer}: alignment={result['mean_alignment']:.4f}{flag}")
    for p in result['per_position']:
        bar = '█' * int(abs(p['cosine']) * 20)
        sign = '+' if p['cosine'] > 0 else '-'
        print(f"    Pos {p['position']}: cos={p['cosine']:+.4f} {sign}{bar}")

## Per-Head Contribution

Which heads are the dominant writers?

In [ ]:
for layer in range(4):
    result = per_head_residual_contribution(model, tokens, layer=layer)
    print(f"  Layer {layer}: dominant=H{result['dominant_head']}")
    for h in result['per_head']:
        bar = '█' * int(h['fraction'] * 40)
        print(f"    H{h['head']}: norm={h['mean_norm']:.4f}, frac={h['fraction']:.2f}, "
              f"align={h['mean_alignment']:+.4f} {bar}")

## Update Magnitude

How much does attention modify the representation?

In [ ]:
for layer in range(4):
    result = attention_update_magnitude(model, tokens, layer=layer)
    flag = ' [LARGE]' if result['is_large_update'] else ''
    print(f"  Layer {layer}: mean_ratio={result['mean_update_ratio']:.4f}{flag}")
    for p in result['per_position']:
        bar = '█' * int(p['update_ratio'] * 20)
        print(f"    Pos {p['position']}: ratio={p['update_ratio']:.4f} {bar}")

## Direction Consistency

Does attention write the same direction at every position?

In [ ]:
for layer in range(4):
    result = attention_direction_consistency(model, tokens, layer=layer)
    flag = ' [CONSISTENT]' if result['is_consistent'] else ''
    print(f"  Layer {layer}: sim={result['mean_direction_similarity']:.4f}{flag}")

## Residual Contribution Summary

In [ ]:
result = attention_residual_summary(model, tokens)
for p in result['per_layer']:
    flag = ' [REINFORCING]' if p['is_reinforcing'] else ''
    print(f"  Layer {p['layer']}: align={p['mean_alignment']:.4f}, "
          f"update_ratio={p['mean_update_ratio']:.4f}{flag}")